# Mutation Set Overlap: Within-Patient vs Cross-Patient

Answers the question: are the mutations we find in a patient's **tumor** and **normal** channels
essentially the same set (as expected for germline variants), or are there tissue-specific differences?

Compares:
- **Same patient, tumor vs normal** — expected high overlap if germline
- **Same patient, same tissue type** — sanity check (should be identical or very close)
- **Cross-patient** — expected low overlap (patient-specific rare variants)

Outputs per plex:
- Pairwise Jaccard similarity distributions by pair type
- Mutation overlap heatmap (all channels)
- Summary: median within-patient vs cross-patient overlap
- List of mutations unique to tumor or normal within the same patient

In [ ]:
import os
import re
import itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

# ── CONFIG ────────────────────────────────────────────────────────────────────
PLEX_LIST_PATH = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
REPO_DIR       = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP        = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META       = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
FASTA_DIR      = "/scratch/leduc.an/AAS_Evo/FASTA/per_sample"
OUT_DIR        = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/validation_multi"

N_PLEXES = 5

os.makedirs(OUT_DIR, exist_ok=True)

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
}

with open(PLEX_LIST_PATH) as f:
    plex_ids = [l.strip() for l in f if l.strip()][:N_PLEXES]

tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})

def short_label(plex_id):
    parts = plex_id.split("_")
    return f"{parts[1]}\n{parts[3]}" if len(parts) >= 4 else plex_id[:20]

print(f"Loaded {len(plex_ids)} plexes, TMT map {len(tmt):,} rows, GDC meta {len(gdc):,} rows")

In [ ]:
# ── BUILD PER-CHANNEL MUTATION SETS FROM FASTAs ───────────────────────────────
# For each plex, for each channel that has a FASTA:
#   mutation_set[plex_id][channel] = frozenset of (accession, swap) tuples
# Also record metadata: channel → (case_submitter_id, sample_type)

def build_plex_mutation_sets(plex_id):
    """Returns (channel_mutations, channel_meta) for one plex.
    channel_mutations: dict channel → frozenset of (acc, swap)
    channel_meta:      dict channel → {'case': str, 'sample_type': str, 'uuid': str}
    """
    plex_tmt = (tmt[tmt["run_metadata_id"] == plex_id]
                [["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates())
    plex_tmt = plex_tmt[~plex_tmt["case_submitter_id"].str.lower()
                        .isin(["ref","reference","pooled","pool","nan"])]
    plex_tmt["channel"] = plex_tmt["tmt_channel"].map(TMT_CHANNEL_MAP)

    plex_meta = plex_tmt.merge(
        gdc[["gdc_file_id","case_submitter_id","sample_type"]],
        on=["case_submitter_id","sample_type"], how="left")

    channel_meta = {}
    for _, row in plex_meta.iterrows():
        ch = row.get("channel")
        if pd.isna(ch): continue
        channel_meta[ch] = {
            "case":        row.get("case_submitter_id", ""),
            "sample_type": row.get("sample_type", ""),
            "uuid":        row.get("gdc_file_id", ""),
        }

    channel_mutations = {}
    for ch, meta in channel_meta.items():
        uuid = meta["uuid"]
        if pd.isna(uuid) or not uuid: continue
        fasta_path = os.path.join(FASTA_DIR, f"{uuid}_mutant.fasta")
        if not os.path.exists(fasta_path): continue
        muts = set()
        with open(fasta_path) as f:
            for line in f:
                if not line.startswith(">"): continue
                parts = line[1:].strip().split("|")
                if len(parts) >= 4 and parts[0] == "mut":
                    muts.add((parts[1], parts[3]))
        channel_mutations[ch] = frozenset(muts)

    return channel_mutations, channel_meta


print("Building per-channel mutation sets from FASTAs...")
plex_channel_mutations = {}  # plex_id → channel_mutations dict
plex_channel_meta      = {}  # plex_id → channel_meta dict

for plex_id in plex_ids:
    cm, meta = build_plex_mutation_sets(plex_id)
    plex_channel_mutations[plex_id] = cm
    plex_channel_meta[plex_id]      = meta
    n_with_fasta = len(cm)
    total_muts   = sum(len(v) for v in cm.values())
    print(f"  {plex_id[:50]:50s}  channels with FASTA: {n_with_fasta}  "
          f"total mutations: {total_muts:,}")

print("Done.")

In [ ]:
# ── COMPUTE PAIRWISE JACCARD SIMILARITY ───────────────────────────────────────
# For every pair of channels within a plex, compute:
#   Jaccard = |A ∩ B| / |A ∪ B|
# Classify pairs:
#   'within_patient_diff_type' — same case, different sample_type (tumor vs normal)
#   'within_patient_same_type' — same case, same sample_type (shouldn't occur often)
#   'cross_patient'            — different cases

def jaccard(a, b):
    if not a and not b: return np.nan
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union > 0 else np.nan

def classify_pair(meta_a, meta_b):
    if meta_a["case"] != meta_b["case"]:
        return "cross_patient"
    # Same patient
    if meta_a["sample_type"] != meta_b["sample_type"]:
        return "within_patient_diff_type"
    return "within_patient_same_type"


all_pairs = []  # list of dicts, one per channel pair

for plex_id in plex_ids:
    cm   = plex_channel_mutations[plex_id]
    meta = plex_channel_meta[plex_id]
    channels = sorted(cm.keys())

    for ch_a, ch_b in itertools.combinations(channels, 2):
        j = jaccard(cm[ch_a], cm[ch_b])
        n_shared = len(cm[ch_a] & cm[ch_b])
        pair_type = classify_pair(meta.get(ch_a, {}), meta.get(ch_b, {}))

        all_pairs.append({
            "plex_id":     plex_id,
            "ch_a":        ch_a,
            "ch_b":        ch_b,
            "case_a":      meta.get(ch_a, {}).get("case", ""),
            "case_b":      meta.get(ch_b, {}).get("case", ""),
            "stype_a":     meta.get(ch_a, {}).get("sample_type", ""),
            "stype_b":     meta.get(ch_b, {}).get("sample_type", ""),
            "n_muts_a":    len(cm[ch_a]),
            "n_muts_b":    len(cm[ch_b]),
            "n_shared":    n_shared,
            "n_unique_a":  len(cm[ch_a] - cm[ch_b]),
            "n_unique_b":  len(cm[ch_b] - cm[ch_a]),
            "jaccard":     j,
            "pair_type":   pair_type,
        })

pairs_df = pd.DataFrame(all_pairs)

print(f"Total channel pairs: {len(pairs_df):,}")
print(pairs_df["pair_type"].value_counts().to_string())
print()
print("Jaccard similarity by pair type:")
print(pairs_df.groupby("pair_type")["jaccard"].describe().round(3).to_string())

In [ ]:
# ── PLOT 1: Jaccard distribution by pair type — one panel per plex ────────────
# Within-patient tumor/normal pairs shown in orange.
# Cross-patient pairs shown in blue.
# If germline: orange bars should cluster near 1.0; blue bars near 0.

PAIR_COLORS = {
    "within_patient_diff_type": "#e8703a",
    "within_patient_same_type": "#2ca02c",
    "cross_patient":            "#4878d0",
}
PAIR_LABELS = {
    "within_patient_diff_type": "Same patient, diff tissue (tumor vs normal)",
    "within_patient_same_type": "Same patient, same tissue",
    "cross_patient":            "Cross-patient",
}

n = len(plex_ids)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), sharey=False)
if n == 1: axes = [axes]

bins = np.linspace(0, 1, 41)

for ax, plex_id in zip(axes, plex_ids):
    plex_pairs = pairs_df[pairs_df["plex_id"] == plex_id]
    plotted_any = False

    for ptype in ["cross_patient", "within_patient_same_type", "within_patient_diff_type"]:
        grp = plex_pairs[plex_pairs["pair_type"] == ptype]["jaccard"].dropna()
        if len(grp) == 0: continue
        ax.hist(grp, bins=bins,
                color=PAIR_COLORS[ptype], alpha=0.7, edgecolor="white", linewidth=0.3,
                label=f"{PAIR_LABELS[ptype]} (n={len(grp)}, med={grp.median():.3f})")
        plotted_any = True

    ax.set_xlabel("Jaccard similarity (shared mutations / union)", fontsize=9)
    ax.set_ylabel("Number of pairs", fontsize=9)
    ax.set_title(short_label(plex_id), fontsize=10)
    ax.set_xlim(0, 1)
    if plotted_any:
        ax.legend(fontsize=7, loc="upper center")

fig.suptitle(
    "Mutation set overlap (Jaccard similarity) by pair type\n"
    "High orange = germline variants consistent across tissues; "
    "Low blue = patient-specific variants",
    fontsize=11, y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "mutation_overlap_jaccard_distributions.pdf"), bbox_inches="tight")
plt.show()
print(f"Saved: mutation_overlap_jaccard_distributions.pdf")

In [ ]:
# ── PLOT 2: Per-plex mutation overlap heatmap ─────────────────────────────────
# Rows/cols = channels, sorted by (case, sample_type).
# Cell colour = Jaccard similarity.
# Channels from the same patient are adjacent → block-diagonal structure expected
# if variants are patient-specific; high off-diagonal values = shared variants.

pdf_path = os.path.join(OUT_DIR, "mutation_overlap_heatmaps.pdf")
with PdfPages(pdf_path) as pdf:
    for plex_id in plex_ids:
        cm   = plex_channel_mutations[plex_id]
        meta = plex_channel_meta[plex_id]
        if not cm: continue

        # Sort channels: same patient together, tumor before normal
        def sort_key(ch):
            m = meta.get(ch, {})
            st = m.get("sample_type", "")
            # tumor first
            st_order = 0 if "tumor" in st.lower() else 1
            return (m.get("case", ""), st_order, ch)

        channels = sorted(cm.keys(), key=sort_key)
        n_ch = len(channels)

        # Build Jaccard matrix
        mat = np.full((n_ch, n_ch), np.nan)
        for i, ch_a in enumerate(channels):
            for j, ch_b in enumerate(channels):
                if i == j:
                    mat[i, j] = 1.0
                elif j > i:
                    mat[i, j] = mat[j, i] = jaccard(cm[ch_a], cm[ch_b])

        # Tick labels: channel + sample type abbreviation + case
        def tick_label(ch):
            m = meta.get(ch, {})
            st = m.get("sample_type", "")
            st_abbr = "T" if "tumor" in st.lower() else ("N" if "normal" in st.lower() else "?")
            case = m.get("case", "")[-7:]  # last 7 chars to keep short
            return f"{ch}\n{st_abbr} {case}"

        labels = [tick_label(ch) for ch in channels]

        fig, ax = plt.subplots(figsize=(max(6, n_ch * 0.85), max(5, n_ch * 0.75)))
        im = ax.imshow(mat, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
        fig.colorbar(im, ax=ax, label="Jaccard similarity")

        ax.set_xticks(range(n_ch))
        ax.set_yticks(range(n_ch))
        ax.set_xticklabels(labels, fontsize=7, rotation=45, ha="right")
        ax.set_yticklabels(labels, fontsize=7)

        # Annotate each cell with Jaccard value
        for i in range(n_ch):
            for j in range(n_ch):
                v = mat[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                            fontsize=6, color="black" if 0.3 < v < 0.8 else "white")

        # Draw boxes around same-patient groups
        case_groups = defaultdict(list)
        for idx, ch in enumerate(channels):
            case_groups[meta.get(ch, {}).get("case", "")].append(idx)
        for case, idxs in case_groups.items():
            if len(idxs) > 1:
                lo, hi = min(idxs), max(idxs)
                rect = plt.Rectangle((lo - 0.5, lo - 0.5), hi - lo + 1, hi - lo + 1,
                                     fill=False, edgecolor="#e8703a", linewidth=2)
                ax.add_patch(rect)

        ax.set_title(f"{plex_id}\nMutation set Jaccard similarity — orange boxes = same patient",
                     fontsize=9)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved: {pdf_path}")

In [ ]:
# ── TABLE: Mutations unique to tumor vs unique to normal within same patient ───
# For each within-patient tumor/normal pair:
#   - mutations in tumor ONLY (somatic candidates or calling artefacts)
#   - mutations in normal ONLY (also artefacts, or low coverage in tumor)
#   - mutations shared (true germline)
# This directly shows whether the exclusion in validation_2.ipynb would matter.

rows_unique = []
for plex_id in plex_ids:
    cm   = plex_channel_mutations[plex_id]
    meta = plex_channel_meta[plex_id]

    plex_pairs_wp = pairs_df[
        (pairs_df["plex_id"] == plex_id) &
        (pairs_df["pair_type"] == "within_patient_diff_type")
    ]

    for _, row in plex_pairs_wp.iterrows():
        ch_a, ch_b = row["ch_a"], row["ch_b"]
        st_a = meta.get(ch_a, {}).get("sample_type", "")
        st_b = meta.get(ch_b, {}).get("sample_type", "")

        # Label which is tumor and which is normal
        if "tumor" in st_a.lower():
            ch_tumor, ch_normal = ch_a, ch_b
        else:
            ch_tumor, ch_normal = ch_b, ch_a

        muts_t = cm.get(ch_tumor,  frozenset())
        muts_n = cm.get(ch_normal, frozenset())
        shared      = muts_t & muts_n
        tumor_only  = muts_t - muts_n
        normal_only = muts_n - muts_t

        rows_unique.append({
            "plex_id":         plex_id,
            "case":            row["case_a"],
            "ch_tumor":        ch_tumor,
            "ch_normal":       ch_normal,
            "n_total_tumor":   len(muts_t),
            "n_total_normal":  len(muts_n),
            "n_shared":        len(shared),
            "n_tumor_only":    len(tumor_only),
            "n_normal_only":   len(normal_only),
            "jaccard":         row["jaccard"],
            "pct_shared_tumor":  100 * len(shared) / len(muts_t)  if muts_t  else np.nan,
            "pct_shared_normal": 100 * len(shared) / len(muts_n) if muts_n else np.nan,
        })

unique_df = pd.DataFrame(rows_unique)

if len(unique_df) == 0:
    print("No within-patient tumor/normal pairs found with FASTAs in these plexes.")
    print("This means normal-tissue BAMs were not processed through variant calling,")
    print("so normal channels have no per-sample FASTAs and are absent from the ratio calculation entirely.")
else:
    print("Within-patient tumor vs normal mutation overlap:")
    print(unique_df[["plex_id","case","ch_tumor","ch_normal",
                     "n_total_tumor","n_total_normal","n_shared",
                     "n_tumor_only","n_normal_only",
                     "pct_shared_tumor","pct_shared_normal","jaccard"]]
          .to_string(index=False))
    print()
    print(f"Median Jaccard (tumor vs normal, same patient): {unique_df['jaccard'].median():.3f}")
    print(f"Median % of tumor mutations shared with normal: {unique_df['pct_shared_tumor'].median():.1f}%")
    print(f"Median % of normal mutations shared with tumor: {unique_df['pct_shared_normal'].median():.1f}%")
    print()
    print("Implication for same-patient exclusion:")
    n_with_unique = (unique_df["n_tumor_only"] > 0).sum()
    print(f"  {n_with_unique}/{len(unique_df)} tumor channels have mutations NOT in their matched normal")
    print(f"  → For these mutations, the normal channel would land in not_have and should be excluded")

    tsv_path = os.path.join(OUT_DIR, "within_patient_tumor_normal_overlap.tsv")
    unique_df.to_csv(tsv_path, sep="\t", index=False)
    print(f"\nSaved: {tsv_path}")

In [ ]:
# ── PLOT 3: Summary — within-patient vs cross-patient Jaccard, all plexes ─────
# Box plot: x = pair type, y = Jaccard similarity.
# If germline dominant: within-patient pairs should have Jaccard close to 1;
# cross-patient pairs should be near 0 (patient-specific variants).

pair_types_ordered = ["within_patient_diff_type", "within_patient_same_type", "cross_patient"]
pair_types_present = [pt for pt in pair_types_ordered if pt in pairs_df["pair_type"].values]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: box plot across all plexes pooled
ax = axes[0]
data_by_type = [pairs_df[pairs_df["pair_type"] == pt]["jaccard"].dropna().values
                for pt in pair_types_present]
bp = ax.boxplot(
    data_by_type,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker=".", markersize=3, alpha=0.3),
    medianprops=dict(color="#e74c3c", linewidth=2),
)
for patch, pt in zip(bp["boxes"], pair_types_present):
    patch.set_facecolor(PAIR_COLORS[pt])
    patch.set_alpha(0.6)
tick_labels = [
    f"{PAIR_LABELS[pt]}\n(n={len(pairs_df[pairs_df['pair_type']==pt]):,}, "
    f"med={pairs_df[pairs_df['pair_type']==pt]['jaccard'].median():.3f})"
    for pt in pair_types_present
]
ax.set_xticks(range(1, len(pair_types_present) + 1))
ax.set_xticklabels(tick_labels, fontsize=8)
ax.set_ylabel("Jaccard similarity", fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.set_title("Mutation overlap by pair type\n(all plexes pooled)", fontsize=10)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)

# Right: scatter — n_shared vs n_unique_a, coloured by pair type
ax = axes[1]
for pt in pair_types_present:
    grp = pairs_df[pairs_df["pair_type"] == pt]
    ax.scatter(grp["n_shared"], grp["jaccard"],
               color=PAIR_COLORS[pt], alpha=0.4, s=10,
               label=PAIR_LABELS[pt])
ax.set_xlabel("N shared mutations", fontsize=10)
ax.set_ylabel("Jaccard similarity", fontsize=10)
ax.set_title("Shared mutation count vs Jaccard", fontsize=10)
ax.legend(fontsize=7)

fig.suptitle("Mutation set overlap: within-patient vs cross-patient", fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "mutation_overlap_summary.pdf"), bbox_inches="tight")
plt.show()
print(f"Saved: mutation_overlap_summary.pdf")

In [ ]:
# ── BONUS: Which mutations are shared across the most patients? ───────────────
# High cross-patient sharing = common germline polymorphisms (should have been
# filtered by gnomAD AF > 0.01 but might slip through).
# Low cross-patient sharing = rare patient-specific variants (good).

print("Mutations shared across the most distinct patients (all plexes)...\n")

# Build: mutation → set of case IDs that have it
mutation_to_cases = defaultdict(set)
for plex_id in plex_ids:
    cm   = plex_channel_mutations[plex_id]
    meta = plex_channel_meta[plex_id]
    for ch, muts in cm.items():
        case = meta.get(ch, {}).get("case", "")
        if not case: continue
        for m in muts:
            mutation_to_cases[m].add(case)

mut_freq = pd.DataFrame([
    {"accession": acc, "swap": swap, "n_patients": len(cases)}
    for (acc, swap), cases in mutation_to_cases.items()
]).sort_values("n_patients", ascending=False)

n_total_patients = sum(len(plex_channel_meta[p]) for p in plex_ids)

print(f"Total unique mutations across all plexes: {len(mut_freq):,}")
print(f"Mutations found in >1 patient: {(mut_freq['n_patients'] > 1).sum():,}")
print(f"Mutations found in >3 patients: {(mut_freq['n_patients'] > 3).sum():,}")
print()
print("Top 20 most-shared mutations:")
print(mut_freq.head(20).to_string(index=False))

# Plot distribution of patient frequency
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mut_freq["n_patients"], bins=range(1, mut_freq["n_patients"].max() + 2),
        color="#4878d0", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Number of distinct patients sharing mutation", fontsize=10)
ax.set_ylabel("Number of mutations", fontsize=10)
ax.set_title("How patient-specific are the mutations in our FASTAs?", fontsize=10)
ax.set_yscale("log")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "mutation_patient_frequency.pdf"), bbox_inches="tight")
plt.show()

mut_freq.to_csv(os.path.join(OUT_DIR, "mutation_patient_frequency.tsv"), sep="\t", index=False)
print(f"\nSaved: mutation_patient_frequency.pdf + .tsv")